[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andreas-he/ai-safety-challenges/blob/main/challenges/2026-05-07-attribution-patching/challenge.ipynb)

# Attribution Patching — Fast Causal Analysis with One Backward Pass

**Day 19 — Mech Interp | short-coding | ~30 min**

**Why this matters:** Attribution patching is the scalable first-pass tool for circuit discovery — instead of O(n) forward passes (one per component), a single backward pass lets you rank every attention head simultaneously. This is the method you'll reach for first when investigating causal structure in the SAIGE inoculation/split-personality project and in ARENA mech interp exercises.

---

Mechanistic interpretability needs to identify *which* components (heads, MLPs, token positions) drive a model's behavior. **Activation patching** answers this definitively by physically swapping activations between a clean and a corrupted run — but at O(n) forward-pass cost. **Attribution patching** (Nanda et al., 2022) cuts this to one forward + one backward pass via a first-order Taylor expansion:

```
score ≈ ∇_a L · (a_patch − a_clean)
```

The gradient `∇_a L` tells you the local slope of the loss w.r.t. each activation dimension. `(a_patch − a_clean)` is the perturbation. Their dot product approximates "if we swapped in the patched activation, how much would the loss change?" — **all in one pass**.


## Setup


In [ ]:
import numpy as np
np.random.seed(42)

# All tasks use pure numpy — no PyTorch required.
# Shape conventions used throughout:
#   d             = feature/hidden dimension
#   n_components  = number of components being analyzed (e.g. n_layers × n_heads)


## Task 1: Single Component Attribution Score

**Formula:** `score = ∇_a L · (a_patch − a_clean)` (standard dot product)

- `grad_a` (shape `(d,)`) — gradient of the loss w.r.t. the component's activation. Tells you how the loss responds to small changes in each activation dimension.
- `a_clean` (shape `(d,)`) — the activation from the clean (baseline) run.
- `a_patch` (shape `(d,)`) — the activation from the corrupted/patched run.

A **positive** score means patching this component increases loss (it was helping the clean behavior). A **negative** score means it was hurting it. Large absolute value = important component.


In [ ]:
def attribution_score(grad_a: np.ndarray, a_clean: np.ndarray, a_patch: np.ndarray) -> float:
    """
    Compute the attribution patching score for a single component.
    Uses a first-order Taylor expansion: score ≈ grad_a · (a_patch - a_clean)

    Args:
        grad_a:  gradient of loss w.r.t. the activation, shape (d,)
        a_clean: baseline (clean run) activation, shape (d,)
        a_patch: counterfactual (corrupted run) activation, shape (d,)

    Returns:
        float: scalar attribution score
    """
    raise NotImplementedError


### Tests


In [ ]:
# delta = [1.0, -1.0, 2.0]; dot([1.0, -2.0, 0.5], [1.0, -1.0, 2.0]) = 1 + 2 + 1 = 4.0
grad = np.array([1.0, -2.0, 0.5])
a_clean_t1 = np.array([1.0, 1.0, 1.0])
a_patch_t1 = np.array([2.0, 0.0, 3.0])
assert np.isclose(attribution_score(grad, a_clean_t1, a_patch_t1), 4.0), \
    f"Expected 4.0, got {attribution_score(grad, a_clean_t1, a_patch_t1)}"

# Negative score: patch moves activation against gradient — loss goes down
grad2 = np.array([1.0, 1.0])
a_c2, a_p2 = np.array([0.0, 0.0]), np.array([-1.0, -1.0])
assert np.isclose(attribution_score(grad2, a_c2, a_p2), -2.0)

print("Task 1 ✓")


## Task 2: Batched Attribution

In practice, you analyze *all* components at once. Given `n_components` components, each with its own gradient and activation pair, compute all scores simultaneously — **no Python loops**.

`grads`, `a_clean`, `a_patch` all have shape `(n_components, d)`. For each `i`, compute `grads[i] · (a_patch[i] - a_clean[i])`.

*Hint: think element-wise multiply then sum, or `np.einsum`.*


In [ ]:
def batched_attribution(grads: np.ndarray, a_clean: np.ndarray, a_patch: np.ndarray) -> np.ndarray:
    """
    Vectorized attribution scores for all components simultaneously.

    Args:
        grads:   shape (n_components, d) — loss gradient w.r.t. each activation
        a_clean: shape (n_components, d) — clean activations
        a_patch: shape (n_components, d) — patched activations

    Returns:
        np.ndarray shape (n_components,): score per component. No Python loops.
    """
    raise NotImplementedError


### Tests


In [ ]:
grads_b = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])  # (3, 2)
a_clean_b = np.zeros((3, 2))
a_patch_b = np.array([[2.0, 3.0], [2.0, 3.0], [2.0, 3.0]])
# Component 0: 1*2 + 0*3 = 2.0 | Component 1: 0*2 + 1*3 = 3.0 | Component 2: 1*2 + 1*3 = 5.0
result_b = batched_attribution(grads_b, a_clean_b, a_patch_b)
assert result_b.shape == (3,), f"Expected shape (3,), got {result_b.shape}"
assert np.allclose(result_b, [2.0, 3.0, 5.0]), f"Expected [2., 3., 5.], got {result_b}"

# Consistency check with Task 1
for i in range(3):
    s = attribution_score(grads_b[i], a_clean_b[i], a_patch_b[i])
    assert np.isclose(s, result_b[i]), "Batched result inconsistent with per-component score"

print("Task 2 ✓")


## Task 3: Rank Components

Attribution patching gives you a score per component. To find the *circuit*, you need to rank them — by **absolute magnitude** (not raw score), because:

- Large positive score → component strongly promotes the clean behavior (patching it hurts)
- Large negative score → component suppresses the clean behavior (also important!)

Return the indices of the `top_k` most important components, sorted by `|score|` descending.


In [ ]:
def rank_components(scores: np.ndarray, top_k: int) -> np.ndarray:
    """
    Identify the most important components by attribution magnitude.

    Args:
        scores: shape (n_components,) — one score per component
        top_k:  number of top components to return

    Returns:
        np.ndarray shape (top_k,): component indices sorted by |score| descending
    """
    raise NotImplementedError


### Tests


In [ ]:
scores_t3 = np.array([0.5, -3.0, 1.2, -0.1, 2.0])
top_idx = rank_components(scores_t3, top_k=3)
assert len(top_idx) == 3
assert top_idx[0] == 1, f"Expected index 1 (|-3.0|=3.0 is largest), got {top_idx[0]}"
assert top_idx[1] == 4, f"Expected index 4 (|2.0|=2.0), got {top_idx[1]}"
assert top_idx[2] == 2, f"Expected index 2 (|1.2|=1.2), got {top_idx[2]}"

# Full pipeline: fake 12-head, 64-dim transformer layer
rng = np.random.default_rng(42)
n_heads, d_head = 12, 64
fake_grads = rng.standard_normal((n_heads, d_head))
fake_clean = rng.standard_normal((n_heads, d_head))
fake_patch = rng.standard_normal((n_heads, d_head))

all_scores = batched_attribution(fake_grads, fake_clean, fake_patch)
top3 = rank_components(all_scores, top_k=3)
assert len(top3) == 3
assert top3[0] == np.argmax(np.abs(all_scores)), "Top-1 head should have the largest |score|"

print("Task 3 ✓ — all tests passed!")
print(f"\nFull pipeline: top 3 heads are {top3}")
print(f"  Attribution scores: {all_scores[top3].round(3)}")


---

**Key insight:** Attribution patching trades O(n) forward passes for one forward + one backward pass, using the gradient as a linear proxy for causal effect. The approximation is exact when activation → loss is linear — and empirically it ranks the same top components as full activation patching, making it the standard first pass before expensive full-patching verification.


<details><summary><b>Solution</b></summary>

```python
def attribution_score(grad_a, a_clean, a_patch):
    return np.dot(grad_a, a_patch - a_clean)

def batched_attribution(grads, a_clean, a_patch):
    # Element-wise multiply then sum along feature dim d
    return np.sum(grads * (a_patch - a_clean), axis=1)
    # Equivalently: np.einsum('nd,nd->n', grads, a_patch - a_clean)

def rank_components(scores, top_k):
    # argsort gives ascending; [::-1] reverses; [:top_k] takes first top_k
    return np.argsort(np.abs(scores))[::-1][:top_k]
```

**Why the formula works:** `∇_a L` is the local slope of the loss surface at the clean activation. `(a_patch − a_clean)` is the step you're approximating. Their dot product is a first-order Taylor expansion — the same math as gradient descent, applied to attribution instead of optimization. Each dimension contributes `grad[i] × delta[i]`; summing gives the total estimated loss change.

**When it breaks:** Large curvature (second-derivative effects dominate), early training (steep loss landscapes), or large `||a_patch − a_clean||`. In practice: use attribution patching to shortlist the top 5–10 heads, then verify with full activation patching on that shortlist.

</details>
